## simple_triton example

This notebook illustrates how to use the simple_triton package to perform inference on tiles from a whole-slide image using a Triton inference server.

Notes for running:
- Run this notebook in a container with `--network=host` so that it can reach the Triton container
- Mount the EfficientNetV2S.tensorflow savedmodel directory to the triton container
- Load the model (below)

In [ ]:
# install large_image with tile sources
!apt update
!apt install -y python3-openslide openslide-tools
!pip install ../../histomics_stream 'large_image[tiff,openslide]' \
  scikit_image --find-links https://girder.github.io/large_image_wheels

# install simple_triton
!pip install ../../simple_triton

# install mil
!pip install ../../mil

## Create a histomics stream study

In [ ]:
import histomics_stream as hs
from mil.io.utils import study
import os

# slide parameters
batch = 64
magnification = 20
tile = 224
overlap = 0
chunk = 1792
mask_threshold=0.5
wsi_path = "/tf/notebooks/TCGA-AN-A0G0-01Z-00-DX1.BE0BB5DF-DEDA-48D8-B5D8-2735C767F28F.svs"
mask_path = "/tf/notebooks/TCGA-AN-A0G0-01Z-00-DX1.BE0BB5DF-DEDA-48D8-B5D8-2735C767F28F.mask.png"

# create a two-slide histomics-stream study
hs_study = study([(wsi_path, mask_path), (wsi_path, mask_path)],
                 t=(tile, tile),
                 chunk=(tile, tile),
                 target=20,
                 source="exact")

## Set parameters and load model

Parameters for this example include parameters for reading from the whole-slide image (magnification, tile size, tile overlap, mask file), the inference server (address), the model (model name, input/output dimensions and type), and the inference client.

We load the model and verify that the model state is "READY".

In [ ]:
import json
from google.protobuf.json_format import MessageToDict
import numpy as np
import tritonclient.grpc as grpcclient

# triton parameters
url = "localhost:8001"  # url for grpc access to tirton server
input_dtype = np.float32  # set input data type
model_name = "EfficientNetV2S.tensorflow"  # set model name
dimension_output = 1280

# create triton client
client = grpcclient.InferenceServerClient(url=url, verbose=True)

# load tensorflow model
updated = {'platform': 'tensorflow_savedmodel',
           'input': [{'name': 'input_2', 'dataType': 'TYPE_FP32', 'dims': [f'{tile}', f'{tile}', '3']}],
           'output': [{'name': 'avg_pool', 'dataType': 'TYPE_FP32', 'dims': [f'{dimension_output}']}],
           'maxBatchSize': 256}
client.load_model("EfficientNetV2S.tensorflow", config=json.dumps(updated))

# check readiness
client.get_model_repository_index()

# deleting the client in main prevents conflicts with child process clients
del client

## Run the inference

In [ ]:
from simple_triton.sharded_tiles import histomics_stream_inference
from simple_triton.submitter import analyze
import time

# inference parameters
limit = 10  # limit on number of pending requests per worker
workers = 32  # total number of Submitter workers
verbose = True  # set verbose as False

# start timer
start = time.time()

# inference
features, results, times = histomics_stream_inference(hs_study, 
                                                      model_name, 
                                                      url="localhost:8001", 
                                                      batch=64, 
                                                      workers=32,
                                                      limit=10)

# display elapsed time
print(f"Total elapsed time: {time.time()-start}")

# analyze performance
analyze(times)